## Учимся получать эмбеддинги из DINOv2*

*Meta признана экстремистской организацией.

1. Рассчитываем cls_token dinov2 для всех ббоксов, полученных с помощью YOLO
2. Складываем их в ChromaDB
3. Папку ChromaDB кладём в s3

In [1]:
from tqdm import tqdm
from pathlib import Path
import os
from PIL import Image
import chromadb

from sneakers_hse.inference.embedding_model import DINOv2Embedder
from sneakers_hse.data.utils.s3_tools import S3Client

/Users/a.r.makarenko/Documents/hse/sneakers-hse/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
PROJECT_ROOT_PATH = Path('/Users/a.r.makarenko/Documents/hse/sneakers-hse')
PATH_TO_YOLO_OUTPUT = PROJECT_ROOT_PATH / 'data/03_yolo_preprocessed/search-dataset-images'

# соберём все нужные файлы заранее
all_files = []
for root, _, files in os.walk(PATH_TO_YOLO_OUTPUT):
    for file in files:
        local_path = os.path.join(root, file)
        relative_path = os.path.relpath(local_path, PATH_TO_YOLO_OUTPUT)
        if not relative_path.endswith(".DS_Store"):
            all_files.append((root, file, relative_path))

len(all_files)

11301

In [3]:
# Инициализируем эмбеддер и векторную БД
embedder = DINOv2Embedder(device='mps')

client = chromadb.PersistentClient(
    path=PROJECT_ROOT_PATH / "chroma_db"
)
collection = client.get_or_create_collection(
    "embeddings",
    metadata={"hnsw:space": "cosine"})


W0503 23:24:07.179000 36608 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [4]:
def embed_bbox(relative_paths: list[str | Path]):
    image_paths = [PATH_TO_YOLO_OUTPUT / relative_path for relative_path in relative_paths]
    imgs = [Image.open(path) for path in image_paths]
    embeddings = embedder.encode_batch(imgs)
    ids = [str(relative_path) for relative_path in relative_paths]
    paths = relative_paths
    classes = [str(relative_path).split("/")[0] for relative_path in relative_paths]
    collection.add(
        ids=ids,
        embeddings=embeddings.tolist(),
        metadatas=[{
            "path": str(relative_path),
            "class": str(relative_path).split("/")[0]
        } for relative_path in relative_paths])
    return [{
                "path": path,
                "class": cls,
                "embedding": emb.tolist()
            } for emb, path, cls in zip(embeddings, paths, classes)]

embed_bbox(['adidas Кеды VL COURT 3/clothing_0_2.jpeg',
            'Nike Кеды Dunk Low Retro/clothing_0_86.jpeg'])

[{'path': 'adidas Кеды VL COURT 3/clothing_0_2.jpeg',
  'class': 'adidas Кеды VL COURT 3',
  'embedding': [-0.04146958142518997,
   -0.00884473416954279,
   -0.02672874741256237,
   -0.002645723056048155,
   0.02280697040259838,
   -0.04448045417666435,
   -0.030424559488892555,
   -0.034934476017951965,
   0.05754683539271355,
   0.0029540371615439653,
   0.008356017991900444,
   0.01380069274455309,
   -0.03808920085430145,
   0.0028538433834910393,
   -0.041374508291482925,
   0.008666593581438065,
   -0.022148720920085907,
   -0.019998840987682343,
   -0.027872245758771896,
   0.01852555386722088,
   -0.056141629815101624,
   0.012183202430605888,
   0.09068237990140915,
   0.0183638297021389,
   0.017451103776693344,
   -0.04438352584838867,
   -0.011609720066189766,
   -0.03737955167889595,
   -0.03242120519280434,
   0.054573751986026764,
   -0.030146421864628792,
   -0.03764621540904045,
   0.025268658995628357,
   -0.021162383258342743,
   -0.11320363730192184,
   -0.004544415

In [5]:
def chunked(iterable, batch_size):
    batch = []
    for item in iterable:
        batch.append(item[2])
        if len(batch) == batch_size:
            yield batch
            batch = []
    if batch:
        yield batch

all_rows = []
# прогресс‑бар с полосой
for relative_paths in tqdm(chunked(iter(all_files), 32), desc="Processing images"):
    all_rows += embed_bbox(relative_paths)

Processing images: 21it [00:26,  1.28s/it]


KeyboardInterrupt: 

In [6]:
import polars as pl

df = pl.DataFrame(all_rows)
df.write_parquet(PROJECT_ROOT_PATH / 'data/04_dinov2_embeddings.parquet.gzip')

In [7]:
from dotenv import load_dotenv

load_dotenv()
s3 = S3Client(aws_access_key_id=os.getenv("AWS_ACCESS_KEY"),
              aws_secret_access_key=os.getenv("AWS_SECRET_KEY"))
s3.upload_folder_to_s3_parallel(
    bucket_name='sneakers-hse-images-test',
    s3_prefix='dinov2/chroma_db', # Путь пишем с учётом модели, т.е. каждой модели будет соответстовать отдельный инстанс ChromaDB
    local_folder=str(PROJECT_ROOT_PATH / 'chroma_db'),
    max_workers=10
)

Total files: 5
Uploaded: dinov2/chroma_db/76efa518-b14e-4991-b2c5-3211233aa8ef/link_lists.bin
Uploaded: dinov2/chroma_db/76efa518-b14e-4991-b2c5-3211233aa8ef/length.bin
Uploaded: dinov2/chroma_db/76efa518-b14e-4991-b2c5-3211233aa8ef/header.bin
Uploaded: dinov2/chroma_db/76efa518-b14e-4991-b2c5-3211233aa8ef/data_level0.bin
Uploaded: dinov2/chroma_db/chroma.sqlite3


### Проверяю, что query работает

In [8]:
image_path = PATH_TO_YOLO_OUTPUT / 'adidas Кеды VL COURT 3/clothing_0_2.jpeg'
img = Image.open(image_path)
embedding = embedder.encode_batch([img])
collection.query(embedding)

{'ids': [['adidas Кеды VL COURT 3/clothing_0_2.jpeg',
   'Vans Кеды Upland/clothing_0_212.jpeg',
   'Vans Кеды Upland/orig_108.jpeg',
   'X-Plode Кеды/orig_124.jpeg',
   'X-Plode Кеды/orig_35.jpeg',
   'Vans Кеды Upland/clothing_0_36.jpeg',
   'Vans Кеды Upland/orig_36.jpeg',
   'Vans Кеды Upland/clothing_0_132.jpeg',
   'X-Plode Кеды/orig_165.jpeg',
   'Vans Кеды Upland/orig_133.jpeg']],
 'embeddings': None,
 'documents': [[None, None, None, None, None, None, None, None, None, None]],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'class': 'adidas Кеды VL COURT 3',
    'path': 'adidas Кеды VL COURT 3/clothing_0_2.jpeg'},
   {'class': 'Vans Кеды Upland',
    'path': 'Vans Кеды Upland/clothing_0_212.jpeg'},
   {'path': 'Vans Кеды Upland/orig_108.jpeg', 'class': 'Vans Кеды Upland'},
   {'class': 'X-Plode Кеды', 'path': 'X-Plode Кеды/orig_124.jpeg'},
   {'class': 'X-Plode Кеды', 'path': 'X-Plode Кеды/orig_35.jpeg'},
   {'class': 'Vans 